In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load the model
# embeddings = HuggingFaceEmbeddings(
#     model_name="Qwen/Qwen3-Embedding-4B",   # ou 4B si tu veux un gros gain de vitesse
#     model_kwargs={
#         "device": "cuda",
#         "trust_remote_code": True,
#     },
#     encode_kwargs={
#         "batch_size": 8,   # teste 4, 8, 16 selon ta VRAM
#         "normalize_embeddings": True,
#         "show_progress_bar": True,
#     },
# )
MODEL = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(model_name=MODEL)
#embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2") #mixedbread-ai/mxbai-embed-large-v1

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6724.77it/s]


In [31]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

client = QdrantClient(
    url="http://localhost:6333",
    grpc_port=6334,
    prefer_grpc=True,
)

vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists("test"):
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
vector_store = QdrantVectorStore(
    client=client,
    collection_name="test",
    embedding=embeddings,
)

In [19]:
from langchain_docling.loader import DoclingLoader
from docling.chunking import HybridChunker
from langchain_docling.loader import ExportType

#C:\Users\crist\perso\master\2\pi\pi\data\Manuals\mds_axis_compensation_en.pdf

FILE_PATH = "./../data/Manuals/mds_axis_compensation_en.pdf"
#FILE_PATH = "https://arxiv.org/pdf/2408.09869"

loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=ExportType.DOC_CHUNKS,
    chunker=HybridChunker(
        tokenizer=MODEL,
        max_tokens=300,
        merge_peers=True,
        repeat_table_header=True,
        omit_header_on_overflow=True,
    ),
)

In [20]:
docs = loader.load()

[INFO] 2026-04-10 15:16:19,519 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-10 15:16:19,521 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-10 15:16:19,550 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-10 15:16:19,552 [RapidOCR] main.py:50: Using C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-10 15:16:19,771 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-10 15:16:19,772 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-10 15:16:19,775 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-04-10 15:16:19,776 [RapidOCR] main.py:50: Using C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\

In [21]:
for d in docs[:10]:
    print(f"- {d.page_content=}")

- d.page_content='Short Description: COMP\n© Copyright ISG Industrielle Steuerungstechnik GmbH STEP, Gropiusplatz 10 D-70563 Stuttgart All rights reserved www.isg-stuttgart.de support@isg-stuttgart.de'
- d.page_content='Legal information\nThis documentation was produced with utmost care. The products and scope of functions described are under continuous development. We reserve the right to revise and amend the documentation at any time and without prior notice.\nNo claims may be made for products which have already been delivered if such claims are based on the specifications, figures and descriptions contained in this documentation.'
- d.page_content='Personnel qualifications\nThis description is solely intended for skilled technicians who were trained in control, automation and drive systems and who are familiar with the applicable standards, the relevant documentation and the machining application.\nIt is absolutely vital to refer to this documentation, the instructions below and th

In [28]:
import uuid
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

texts = [d.page_content for d in docs]
payloads = [d.metadata for d in docs]
ids = [str(uuid.uuid4()) for _ in docs]


model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    trust_remote_code=True,
)

vectors = model.encode_document(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
)

Batches: 100%|██████████| 17/17 [01:14<00:00,  4.40s/it]


In [ ]:
client.upload_collection(
    collection_name="test",
    ids=ids,
    vectors=vectors.tolist(),
    payload=payloads,
    parallel=4,
    max_retries=3,
)

In [32]:
# ----------------------------
# 5) Ingestion dans Qdrant
# ----------------------------
ids = vector_store.add_documents(docs)
print(f"Chunks indexés : {len(ids)}")

Chunks indexés : 129


In [30]:
# ----------------------------
# 6) Test retrieval
# ----------------------------
#query = "Quels sont les objectifs principaux du document ?"
# query = "what is this mode P-COMP-00047 ?"
query = "give me the structure, parameter et functionality for P-COMP-00047"
results = vector_store.similarity_search(query, k=15)

for i, doc in enumerate(results, 1):
    print(f"\n===== RESULT {i} =====")
    print(doc.page_content)
    print(doc.metadata)

    


===== RESULT 1 =====

{'_id': '58cf36bb-4e2c-4ba0-b887-cbfd5d86cc73', '_collection_name': 'test'}

===== RESULT 2 =====

{'_id': '1dad3b59-1c5b-4e01-8bed-6f81beab3332', '_collection_name': 'test'}

===== RESULT 3 =====

{'_id': '692e859f-ce60-4aed-bdf7-672979f25e7f', '_collection_name': 'test'}

===== RESULT 4 =====

{'_id': '9e861ddc-e564-4ee4-9f7d-1d0a51d0a745', '_collection_name': 'test'}

===== RESULT 5 =====

{'_id': '50c150e9-3b0d-4702-a1c4-3525580b7d36', '_collection_name': 'test'}

===== RESULT 6 =====

{'_id': '59d28d95-cae1-478a-a56b-f703f244e1c2', '_collection_name': 'test'}

===== RESULT 7 =====

{'_id': '27bb2585-fef4-4573-a3d2-a3de5cd07be5', '_collection_name': 'test'}

===== RESULT 8 =====

{'_id': '928300ae-1250-41e9-80ea-ff977e7cc762', '_collection_name': 'test'}

===== RESULT 9 =====

{'_id': 'cb59e015-e5b1-4c2a-955b-973c29e8dd38', '_collection_name': 'test'}

===== RESULT 10 =====

{'_id': '1dab87c2-2a68-4aa2-9060-7ccdeee66083', '_collection_name': 'test'}

===== RE

In [6]:
from pprint import pprint

pprint(results[2].page_content)

('3.4.2 Number of elements in the compensation value table (P-COMP-00042)\n'
 'Description, Number of elements in the compensation value table = This '
 'parameter defines the number of entries in the compensation table.. '
 'Description, Number of elements in the compensation value table = This '
 'parameter defines the number of entries in the compensation table.. '
 'Parameter, Number of elements in the compensation value table = '
 'frict_comp.table_entries. Parameter, Number of elements in the compensation '
 'value table = frict_comp.table_entries. Data type, Number of elements in the '
 'compensation value table = UNS16. Data type, Number of elements in the '
 'compensation value table = UNS16. Data range, Number of elements in the '
 'compensation value table = 0 ≤ table_entries ≤ 20. Data range, Number of '
 'elements in the compensation value table = 0 ≤ table_entries ≤ 20. Axis '
 'types, Number of elements in the compensation value table = T, R, S. Axis '
 'types, Number of

In [47]:
from pprint import pprint

pprint(results[1].metadata)

{'_collection_name': 'test',
 '_id': '9ec6799c-faa5-49f4-9e5e-25507270b4a1',
 'dl_meta': {'doc_items': [{'children': [],
                            'content_layer': 'body',
                            'label': 'table',
                            'parent': {'$ref': '#/body'},
                            'prov': [{'bbox': {'b': 409.18267822265625,
                                               'coord_origin': 'BOTTOMLEFT',
                                               'l': 55.41105270385742,
                                               'r': 552.8771362304688,
                                               't': 626.1316223144531},
                                      'charspan': [0, 0],
                                      'page_no': 34}],
                            'self_ref': '#/tables/46'}],
             'headings': ['3.4.1 Friction interpolation mode (P-COMP-00041)'],
             'origin': {'binary_hash': 5388167723037981658,
                        'filename': 'mds_axis_comp